In [ ]:
import time
from collections import deque
from pathlib import Path

import google.genai.types as genai_types
import pandas as pd
from post_processing.io_utils import load_json_from_file
from google import genai
from google.genai.types import HttpOptions
from langchain_core.prompts import PromptTemplate
from tqdm import tqdm

from data_extraction.data_loaders import get_papers
from data_extraction.table_relevance_prompt import EXPRESSION_DETAILS_USER_PROMPT, EXPRESSION_DETAILS_PROMPT
from data_extraction.utils import encode_image
from data_extraction.utils import extract_json_from_text, save_json_to_file
from data_extraction.utils import save_to_txt_file, get_model_version

In [ ]:
client = genai.Client(http_options=HttpOptions(api_version="v1"),
                      api_key="***")
MODEL_NAME = "gemini-2.5-pro"
MODEL_VERSION = get_model_version(MODEL_NAME)
MODEL_VERSION

In [ ]:
def prepare_expression_detail_requests(
        paths_to_images: list[Path],
        papers_df: pd.DataFrame,
        results_dir: Path
) -> tuple[list[dict], list[str]]:
    print("Preparing multi-modal requests for expression detail extraction...")
    requests = []
    processed_image_stems = []

    # Ensure the results directory exists
    results_dir.mkdir(parents=True, exist_ok=True)

    user_prompt_template = PromptTemplate.from_template(EXPRESSION_DETAILS_USER_PROMPT)

    for image_path in paths_to_images:
        try:
            # Extract PMID from the image filename (e.g., "27309616_page_5_table_0.png")
            pmid = image_path.stem.split("_")[0]

            # Find the paper information for the current PMID
            paper_info = papers_df[papers_df["PMID"] == pmid].iloc[0]

            # Format the text part of the user prompt
            user_prompt_text = user_prompt_template.format(
                title=paper_info["Title"],
                abstract=paper_info["Abstract"]
            )

            # Save the text prompt for debugging
            save_to_txt_file(user_prompt_text, results_dir / f"{image_path.stem}_user_prompt.txt")

            # Encode the image
            image_data = encode_image(image_path)

            # Construct the request dictionary for the batch API.
            # This follows a multi-turn format with a system prompt and a multi-modal user prompt.
            request = {
                "contents": [
                    {
                        "role": "user",  # Using 'user' role for system prompt, as per your example
                        "parts": [{"text": EXPRESSION_DETAILS_PROMPT}]
                    },
                    {
                        "role": "user",
                        "parts": [
                            # Part 1: The image data
                            {"inline_data": {
                                "mime_type": "image/png",
                                "data": image_data
                            }},
                            # Part 2: The text prompt
                            {"text": user_prompt_text}
                        ]
                    }
                ]
            }
            requests.append(request)
            processed_image_stems.append(image_path.stem)

        except IndexError:
            print(f"Warning: No paper info found for PMID {pmid} (from {image_path.name}). Skipping.")
        except FileNotFoundError:
            print(f"Warning: Image file not found at {image_path}. Skipping.")
        except Exception as e:
            print(f"An error occurred while processing {image_path.name}: {e}")

    print(f"Prepared {len(requests)} requests for processing.")
    return requests, processed_image_stems

In [ ]:
RESULTS_DIR = Path(f"./details_batching_{MODEL_VERSION}/")
papers_df = get_papers("./new_papers.csv")[["PMID", "Title", "Abstract", "EDAT"]].astype({"PMID": str})
path_to_images = "res/pdf_table_images/"
path_to_images = list(path_to_images.glob("*.png"))
inline_requests, processed_images = prepare_expression_detail_requests(path_to_images, papers_df, RESULTS_DIR)

In [ ]:
BATCH_SIZE = 100
original_batches = [inline_requests[i:i + BATCH_SIZE] for i in range(0, len(inline_requests), BATCH_SIZE)]
original_processed_pmids = [processed_images[i:i + BATCH_SIZE] for i in range(0, len(processed_images), BATCH_SIZE)]
len(original_batches), len(original_processed_pmids), len(inline_requests)

In [ ]:
JOBS_PATH = "jobs_names_details.json"

# jobs_names = load_json_from_file("jobs_names2.json")
jobs_names = []
currently_processing = load_json_from_file("currently_processing.json")
queue = deque(original_batches)

In [ ]:
max_jobs_num = 7
while queue or currently_processing:

    while len(currently_processing) < max_jobs_num and queue:
        batch = queue.popleft()

        try:
            job = client.batches.create(
                model=f"models/{MODEL_NAME}",
                src=batch
            )
        except Exception as e:
            queue.appendleft(batch)
            time.sleep(2)
            print(f"An error occurred while processing batch: {e}")
            break

        print("Job created:", job.name)
        print("Queue length:", len(queue))
        jobs_names.append(str(job.name))
        save_json_to_file(jobs_names, JOBS_PATH)
        currently_processing.append(job.name)
        save_json_to_file(currently_processing, "./currently_processing.json")

    print("Running jobs:", len(currently_processing), "\n")
    if currently_processing:
        time.sleep(240)

    print("Checking currently processed jobs...")
    for currently_processed_job_name in currently_processing.copy():
        currently_processed_job = client.batches.get(name=currently_processed_job_name)
        # print(f"Job {currently_processed_job.name} state: {currently_processed_job.state}")
        if currently_processed_job.state == genai_types.JobState.JOB_STATE_SUCCEEDED:
            currently_processing.remove(currently_processed_job_name)
            save_json_to_file(currently_processing, "./currently_processing.json")
            print(f"Job {currently_processed_job.name} succeeded.")


print("All jobs processed. Final job list saved.")

In [ ]:
job = client.batches.create(
    model=f"models/{MODEL_NAME}",
    src=inline_requests,
)

In [ ]:
def process_expression_detail_batch_results(job, processed_image_stems: list, results_dir: Path, model_version: str):
    if job.state != genai_types.JobState.JOB_STATE_SUCCEEDED:
        print(f"❌ Job did not succeed. Final state: {job.state.name}")
        if hasattr(job, 'error') and job.error:
            print(f"Error: {job.error}")
        return

    print(f"✅ Job finished with state: {job.state.name}")
    print("Processing batch results...")

    # Assumes results are inlined in the destination
    results = job.dest.inlined_responses

    # --- 2. Verify Result Count ---
    if len(results) != len(processed_image_stems):
        print(
            f"⚠️ Warning: Mismatch between result count ({len(results)}) and image stem count ({len(processed_image_stems)}).")

    # --- 3. Process Each Result ---
    for image_stem, result in zip(processed_image_stems, results, strict=True):
        try:
            # Save the entire response object as a JSON for detailed review
            # Protobuf messages are converted to dicts for JSON serialization

            full_response_path = results_dir / f"{image_stem}_{model_version}_full_response.json"
            save_json_to_file(result.model_dump(), full_response_path)

            # Extract the generated text content
            response_text = result.response.candidates[0].content.parts[0].text

            # Save the raw text output (e.g., "27309616..._gemini-2.5-flash.txt")
            raw_output_path = results_dir / f"{image_stem}_{model_version}.txt"
            save_to_txt_file(response_text, raw_output_path)

            # Extract the primary JSON from the text and save it
            try:
                output_json = extract_json_from_text(response_text)
            except Exception as e:
                print(f"Error extracting JSON from response text for {image_stem}: {e}")
                continue

            if output_json:
                json_output_path = results_dir / f"{image_stem}_{model_version}.json"
                save_json_to_file(output_json, json_output_path)
            else:
                print(f"Warning: Could not extract primary JSON for image {image_stem}")

        except (AttributeError, IndexError) as e:
            print(f"Warning: Could not parse result structure for {image_stem}. Error: {e}. Result: {result}")
        except Exception as e:
            print(f"An unexpected error occurred while processing result for {image_stem}: {e}")

for job_name, processed_images_batch in zip(jobs_names, original_processed_pmids):
    job = client.batches.get(name=job_name)
    process_expression_detail_batch_results(
        job=job,
        processed_image_stems=processed_images_batch,
        results_dir=RESULTS_DIR,
        model_version=MODEL_VERSION
    )